# DEA Coastlines rates of change summary tables

This notebook steps through the process to create epoch based summary tables of DEA Coastlines rates of change points for a set of multiple polygon locations (e.g. islands).

Steps:
1. Regenerate DEA Coastlines for the area of interest (DEA Coastlines repo)
2. Recalculate the rates of change for the epoch baselines of interest (DEA Coastlines repo)
3. Calculate custom epoch rates of change from pre-calculated baselines (DEA Notebooks repo)
4. Summarise and tabularise per polygon per epoch rates of change data (DEA Notebooks repo)
5. Update the output geopackage

# Steps 1-2: from DEA Coastlines repo

## Load modules

In [ ]:
# Install DEA Coastlines package

!pip install git+https://github.com/GeoscienceAustralia/dea-coastlines.git@stable

In [ ]:
cd '/home/jovyan/dev/dea-coastlines'

In [ ]:
# Update required packages
!pip install -r requirements.in --quiet

In [ ]:
# Set analysis parameters


# name for output raster
raster_version = 'development_TSRA'


# name for output vector
vector_version = f'development_TSRA_{baseline}_bl'
# name for combined outputs
continental_version = f'development_TSRA_{baseline}_bl'
# path to configs including grid and virtual products
config_path = 'configs/dea_coastlines_config.yaml'

In [ ]:
# Use the DEA Coastlines command line interface tools

# Set analysis parameters

# baseline year to calculate rates of change from
baselines = [2025, 2020, 2015, 2010, 2005, 2000]
# list of iles of interest from config_path
study_areas = [20]#[1,2,3,4,5,6,7,8,9,10,11,12,13,15,16,17,18,19,20]
# names for output vectors
vector_versions = []
# identify tide model
tide_model = 'ensemble'

# Run raster and vector generation for each study area
for study_area in study_areas:
    print(study_area)
    # Generate DEA Coastlines rasters of MNDWI
    !python -m coastlines.raster --config_path {config_path} --study_area {study_area} --raster_version {raster_version} --start_year 1988 --end_year 2025 --tide_model {tide_model}
    for baseline in baselines:
        # name for output vector
        vector_version = f'development_TSRA_{baseline}_bl'
        vector_versions.append(vector_version)
        # Calculate DEA Coastlines shoreline and rates of change vectors
        !python -m coastlines.vector --config_path {config_path} --study_area {study_area} --raster_version {raster_version} --vector_version {vector_version} --start_year 1988 --end_year {baseline} --baseline_year {baseline}
        

In [ ]:
# When complete, combine into single continental outputs
for vector_version in list(set(vector_versions)):
    print(vector_version)
    # name for output dataset
    continental_version = f'development_TSRA_{vector_version.split("_")[-2]}_bl'
    !python -m coastlines.continental --vector_version {vector_version} --continental_version {continental_version} --shorelines True --ratesofchange True --hotspots False --baseline_year {baseline}

In [7]:
# Temp
baselines = [2025, 2020, 2015, 2010, 2005, 2000]

# Steps 3-4: DEA Notebooks repo

In [1]:
cd '/home/jovyan/dev/dea-notebooks'

/home/jovyan/dev/dea-notebooks


## Load modules

In [2]:
import fiona

import pandas as pd
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

from shapely.geometry import box
from sklearn.linear_model import LinearRegression
from dea_tools.coastal import get_coastlines

from scipy.stats import ttest_ind
from scipy.stats import ttest_rel
from scipy.stats import linregress

## Load functions

In [3]:
# Load Coastlines functions

# These are edited functions from dea-coastlines GItHub repo vectors.py

def outlier_mad(points, thresh=3.5):
    """
    Use robust Median Absolute Deviation (MAD) outlier detection
    algorithm to detect outliers. Returns a boolean array with True if
    points are outliers and False otherwise.

    Parameters:
    -----------
    points :
        An n-observations by n-dimensions array of observations
    thresh :
        The modified z-score to use as a threshold. Observations with a
        modified z-score (based on the median absolute deviation) greater
        than this value will be classified as outliers.

    Returns:
    --------
    mask :
        A n-observations-length boolean array.

    References:
    ----------
    Source: https://github.com/joferkington/oost_paper_code/blob/master/utilities.py

    Boris Iglewicz and David Hoaglin (1993), "Volume 16: How to Detect and
    Handle Outliers", The ASQC Basic References in Quality Control:
    Statistical Techniques, Edward F. Mykytka, Ph.D., Editor.
    """
    if len(points.shape) == 1:
        points = points[:, None]
    median = np.median(points, axis=0)
    diff = np.sum((points - median) ** 2, axis=-1)
    diff = np.sqrt(diff)
    med_abs_deviation = np.median(diff)

    modified_z_score = 0.6745 * diff / med_abs_deviation

    return modified_z_score > thresh

def change_regress(
    y_vals,
    x_vals,
    x_labels,
    threshold=3.5,
    detrend_params=None,
    slope_var="slope",
    interc_var="intercept",
    pvalue_var="pvalue",
    stderr_var="stderr",
    outliers_var="outliers",
):
    """
    For a given row in a `pandas.DataFrame`, apply linear regression to
    data values (as y-values) and a corresponding sequence of x-values,
    and return 'slope', 'intercept', 'pvalue', and 'stderr' regression
    parameters.

    Before computing the regression, outliers are identified using a
    robust Median Absolute Deviation (MAD) outlier detection algorithm,
    and excluded from the regression. A list of these outliers will be
    recorded in the output 'outliers' variable.

    Parameters:
    -----------
    x_vals, y_vals : list of numeric values, or nd.array
        A sequence of values to use as the x and y variables
    x_labels : list
        A sequence of strings corresponding to each value in `x_vals`.
        This is used to label any observations that are flagged as
        outliers (often, this can simply be set to the same list
        provided to `x_vals`).
    threshold : float, optional
        The modified z-score to use as a threshold for detecting
        outliers using the MAD algorithm. Observations with a modified
        z-score (based on the median absolute deviation) greater
        than this value will be classified as outliers.
    detrend_params : optional
        Not currently used
    slope, interc_var, pvalue_var, stderr_var : strings, optional
        Strings giving the names to use for each of the output
        regression variables.
    outliers_var : string, optional
        String giving the name to use for the output outlier variable.

    Returns:
    --------
    mask :
        A `pandas.Series` containing regression parameters and lists
        of outliers.

    """

    # Drop invalid NaN rows
    xy_df = np.vstack([x_vals, y_vals]).T
    valid_bool = ~np.isnan(xy_df).any(axis=1)
    xy_df = xy_df[valid_bool]
    valid_labels = x_labels[valid_bool]

    # Check if we have enough data points after removing NaNs (minimum 2)
    if len(xy_df) < 2:
        # Create string of all outliers and invalid NaN rows
        outlier_set = set(x_labels) - set(valid_labels)
        outlier_str = " ".join(map(str, sorted(outlier_set)))
        
        # Return NaN values if insufficient data
        results_dict = {
            slope_var: np.nan,
            interc_var: np.nan,
            pvalue_var: np.nan,
            stderr_var: np.nan,
            outliers_var: outlier_str,
        }
        return pd.Series(results_dict)
    
    # If detrending parameters are provided, apply these to the data to
    # remove the trend prior to running the regression
    if detrend_params:
        xy_df[:, 1] = xy_df[:, 1] - (
            detrend_params[0] * xy_df[:, 0] + detrend_params[1]
        )

    # Remove outliers using MAD
    outlier_bool = outlier_mad(xy_df, thresh=threshold)
    # outlier_bool = outlier_ransac(xy_df)
    xy_df = xy_df[~outlier_bool]
    valid_labels = valid_labels[~outlier_bool]

    # Create string of all outliers and invalid NaN rows
    outlier_set = set(x_labels) - set(valid_labels)
    outlier_str = " ".join(map(str, sorted(outlier_set)))

    # Check again after outlier removal
    if len(xy_df) < 2:
        # Return NaN values if insufficient data
        results_dict = {
            slope_var: np.nan,
            interc_var: np.nan,
            pvalue_var: np.nan,
            stderr_var: np.nan,
            outliers_var: outlier_str,
        }
        return pd.Series(results_dict)
    
    # Compute linear regression
    lin_reg = linregress(x=xy_df[:, 0], y=xy_df[:, 1])

    # Return slope, p-values and list of outlier years excluded from regression
    results_dict = {
        slope_var: np.round(lin_reg.slope, 3),
        interc_var: np.round(lin_reg.intercept, 3),
        pvalue_var: np.round(lin_reg.pvalue, 3),
        stderr_var: np.round(lin_reg.stderr, 3),
        outliers_var: outlier_str,
    }

    return pd.Series(results_dict)

def calculate_regressions(points_gdf1,
                         start,
                         end):
    """
    For each rate of change point along the baseline annual coastline,
    compute linear regression rates of change against both time and
    climate indices.

    Regressions are computed after removing outliers to ensure robust
    results.

    Parameters:
    -----------
    points_gdf : geopandas.GeoDataFrame
        A `geopandas.GeoDataFrame` containing rates of change points
        with 'dist_*' annual movement/distance data.
    start  :  integer (optional)
        A 4 digit integer of the year you wish to begin the epoch calculation.
        For example, the baseline is set as 2020 from the annual_movements function.
        Set `start` = 2015 to calculate rate_change points for the 5 year window
        between 2015 to 2020.
        All years before 'start' are excluded from the rate change calculation.
        Defaults to none, setting the oldest year as the start of the epoch.
    end  :  integer (optional)
        An integer of the year used as baseline for rate change calculations.
        Only used in the variable naming in the returned geopandas.GeoDataFrame

    Returns:
    --------
    points_gdf : geopandas.GeoDataFrame
        A `geopandas.GeoDataFrame` containing rates of change points
        with additional attribute columns:

            'rate_*':  Slope of the regression
            'sig_*':   Significance of the regression
            'se_*':    Standard error of the  regression
            'outl_*':  A list of any outlier years excluded from the
                       regression
    """
    # Copy the input geodataframe
    points_gdf = points_gdf1.copy(deep=True)
    
    # Restrict data to years in datasets
    dist_years = points_gdf.columns[points_gdf.columns.str.contains("dist_")]
    x_years = dist_years.str.replace("dist_", "").astype(int)
    if start:
        x = x_years.where(x_years>=start).dropna().astype('int')
        dist_years = dist_years[-len(x):]
        print(dist_years)
        points_subset = points_gdf[dist_years].copy()
    else:
        points_subset = points_gdf[dist_years].copy()
    
    # Compute coastal change rates by linearly regressing annual
    # movements vs. time
    if start:
        rate_out = points_subset.apply(
            lambda row: change_regress(
                y_vals=row.values.astype(float), x_vals=x, x_labels=x
            ),
            axis=1,
            )
        points_gdf[[f"rate_time_{start}-{end}", f"incpt_time_{start}-{end}", f"sig_time_{start}-{end}", f"se_time_{start}-{end}", f"outl_time_{start}-{end}"]] = (
            rate_out
            )
    
    else:
        rate_out = points_subset.apply(
            lambda row: change_regress(
                y_vals=row.values.astype(float), x_vals=x_years, x_labels=x_years
            ),
            axis=1,
            )
        points_gdf[["rate_time", "incpt_time", "sig_time", "se_time", "outl_time"]] = (
                rate_out
            )
    
    # Copy slope and intercept into points_subset so they can be
    # used to temporally de-trend annual distances
    points_subset[["slope", "intercept"]] = rate_out[["slope", "intercept"]]
    
    # Custom sorting
    if start:
        reg_cols = [f"rate_time_{start}-{end}", f"sig_time_{start}-{end}", f"se_time_{start}-{end}", f"outl_time_{start}-{end}"]
    else:
        reg_cols = ["rate_time", "sig_time", "se_time", "outl_time"]
    
    
    return points_gdf.loc[
        :, [*reg_cols, *dist_years, "angle_mean", "angle_std", "geometry"]
    ]

# Edited functions from dea_tools.coastal
def get_coastlines1(response, layer, bbox: tuple, crs="EPSG:4326") -> gpd.GeoDataFrame:
    """
    Load DEA Coastlines annual shorelines or rates of change points data
    for a provided bounding box using WFS.

    For a full description of the DEA Coastlines dataset, refer to the
    official Geoscience Australia product description:
    /data/product/dea-coastlines

    Parameters
    ----------
    bbox : (xmin, ymin, xmax, ymax), or geopandas object
        Bounding box expressed as a tutple. Alternatively, a bounding
        box can be automatically extracted by suppling a
        geopandas.GeoDataFrame or geopandas.GeoSeries.
    crs : str, optional
        Optional CRS for the bounding box. This is ignored if `bbox`
        is provided as a geopandas object.
    layer : str, optional
        Which DEA Coastlines layer to load. Options include the annual
        shoreline vectors ("shorelines_annual") and the rates of change
        points ("rates_of_change"). Defaults to "shorelines_annual".
    response  :  str
        A file directory path to the annual rates of change DEA Coastlines
        dataset

    Returns
    -------
    gpd.GeoDataFrame
        A GeoDataFrame containing shoreline or point features and
        associated metadata.
    """

    # If bbox is a geopandas object, convert to bbox
    try:
        crs = str(bbox.crs)
        bbox = bbox.total_bounds
    except:
        pass

    # # Query WFS
    # wfs = WebFeatureService(url=WFS_ADDRESS, version="1.1.0")
    # layer_name = f"dea:{layer}"
    # response = wfs.getfeature(
    #     typename=layer_name,
    #     bbox=tuple(bbox) + (crs,),
    #     outputFormat="json",
    # )

    # Load data as a geopandas.GeoDataFrame
    coastlines_gdf = gpd.read_file(response,layer=layer)

    # Clip to extent of bounding box
    extent = gpd.GeoSeries(box(*bbox), crs=crs).to_crs(coastlines_gdf.crs)
    coastlines_gdf = coastlines_gdf.clip(extent)

    # # Optionally drop WMS-specific columns
    # if drop_wms:
    #     coastlines_gdf = coastlines_gdf.loc[:, ~coastlines_gdf.columns.str.contains("wms_")]

    return coastlines_gdf

# After: https://gist.github.com/robbibt/760dcf367be4b98c493e70dd577aca6a
def change_summary(df, sig=0.01, rate=0.31, bias=0.08):

    # Create booleans indicating whether points were significant,
    # greater than the minimum accuracy of the method
    # or highly likely to exhibit non-linear rate change
    sig_bool = df.sig_time <= sig
    # rate_bool = (df.rate_time + bias).abs() >= rate #CP removed as the bias correction has already been applied
    rate_bool = (df.rate_time).abs() >= rate 
    r2_bool = df.r2_improvement >0.5
    
    # Calculate dynamic (linear change) % (sig points greater than min rate)
    stat_dict = {}
    stat_dict['dynamic'] = (sig_bool & rate_bool & ~r2_bool).mean()

    # Calculate dynamic (non-linear change) %
    stat_dict['nonlinear'] = r2_bool.mean()

    # Calculate stable % (non-sig points or less than min rate)
    stat_dict['stable'] = 1.0 - stat_dict['dynamic'] - stat_dict['nonlinear']

    # For each rate of change category, calculate percent greater
    # (prograding) or percent smaller (eroding coasts)

    for rate_cat in [0.0, 0.5, 1.0, 3.0, 5.0]:
        stat_dict[f'eroding_{rate_cat}'] = (
            # sig_bool & rate_bool & (df.rate_time + bias < -rate_cat)).mean() #CP removed as the bias correction has already been applied
            sig_bool & rate_bool & ~r2_bool & (df.rate_time < -rate_cat)).mean()
        stat_dict[f'prograd_{rate_cat}'] = (
            # sig_bool & rate_bool & (df.rate_time + bias > rate_cat)).mean() #CP removed as the bias correction has already been applied
            sig_bool & rate_bool & ~r2_bool & (df.rate_time > rate_cat)).mean()

    return pd.Series(stat_dict)

# Function to calculate R²
def calculate_r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    if ss_tot == 0:
        return 0
    return 1 - (ss_res / ss_tot)

# Function to process a single row
def process_row(row,epoch,dist_cols):
    # Get outlier years
    outlier_string = row[f'outl_time_{epoch}']
    if pd.notna(outlier_string):
        outlier_years = set(str(outlier_string).replace(',', ' ').replace(';', ' ').split())
    else:
        outlier_years = set()
    
    # Extract years and values, excluding outliers
    years = []
    values = []
    
    for col in dist_cols:
        year = col.split('_')[1]
        value = row[col]
        
        if year not in outlier_years and pd.notna(value):
            years.append(int(year))
            values.append(float(value))
    
    # Need at least 4 points for cubic (degree 3)
    if len(values) < 4:
        return pd.Series({
            'r2_linear': None,
            'r2_cubic': None,
            # 'is_nonlinear': None
        })
    
    # Convert to arrays
    years_array = np.array(years)
    years_since_start = years_array - years_array[0]
    y = np.array(values)
    
    try:
        # Fit linear model (degree 1)
        linear_coeffs = np.polyfit(years_since_start, y, 1)
        y_pred_linear = np.polyval(linear_coeffs, years_since_start)
        r2_linear = calculate_r2(y, y_pred_linear)
        
        # Fit cubic model (degree 3)
        cubic_coeffs = np.polyfit(years_since_start, y, 3)
        y_pred_cubic = np.polyval(cubic_coeffs, years_since_start)
        r2_cubic = calculate_r2(y, y_pred_cubic)
        
        # # Determine if non-linear (threshold: 0.1 difference in R²)
        # r2_diff = r2_cubic - r2_linear
        # is_nonlinear = r2_diff > 0.5
        
        return pd.Series({
            'r2_linear': r2_linear,
            'r2_cubic': r2_cubic,
            # 'is_nonlinear': is_nonlinear
        })
        
    except Exception as e:
        return pd.Series({
            'r2_linear': None,
            'r2_cubic': None,
            # 'is_nonlinear': None
        })


## DEA Coastlines definitions

From DEA Coastlines Knowledge Hub [Specifications](https://knowledge.dea.ga.gov.au/data/product/dea-coastlines/?tab=specifications#layers)

	
||Type|Units|Description|
|---|---|---|---|
|uid|String|-|A unique geohash identifier for each point.|
|rate_time|Float|Metres per year|Annual rates of change (in metres per year) calculated by linearly regressing annual shoreline distances against time (excluding outliers). Negative values indicate retreat and positive values indicate growth.|
|sig_time|Float|P-value|Significance (p-value) of the linear relationship between annual shoreline distances and time. Small values (e.g. p-value < 0.01 or 0.05) may indicate a coastline is undergoing consistent coastal change through time.|
|se_time|Float|Metres|Standard error (in metres) of the linear relationship between annual shoreline distances and time. This can be used to generate confidence intervals around the rate of change given by rate_time, e.g. 95% confidence interval = se_time x 1.96|
|outl_time|String|-|Individual annual shoreline are noisy estimators of coastline position that can be influenced by environmental conditions (e.g. clouds, breaking waves, sea spray) or modelling issues (e.g. poor tidal modelling results or limited clear satellite observations). To obtain reliable rates of change, outlier shorelines are excluded using a robust Median Absolute Deviation outlier detection algorithm, and recorded in this column.|
|dist_1990, dist_1991, etc|Float|Metres|Annual shoreline distances (in metres) relative to the most recent baseline shoreline. Negative values indicate that an annual shoreline was located inland of the baseline shoreline. By definition, the most recent baseline column will always have a distance of 0 m.|
|angle_mean, angle_std|Integer|Degrees|The mean angle and standard deviation between the baseline point to all annual shorelines. This data is used to calculate how well shorelines fall along a consistent line; high angular standard deviation indicates that derived rates of change are unlikely to be correct.|
|valid_obs|Integer|-|The total number of valid (i.e. non-outliers, non-missing) annual shoreline observations.|
valid_span|Integer|Years|The maximum number of years between the first and last valid annual shoreline.|
|sce|Float|Metres|Shoreline Change Envelope (SCE). A measure of the maximum change or variability across all annual shorelines, calculated by computing the maximum distance between any two annual shorelines (excluding outliers). This statistic excludes sub-annual shoreline variability.|
|nsm|Float|Metres|Net Shoreline Movement (NSM). The distance between the oldest (1988) and most recent annual shoreline (excluding outliers). Negative values indicate the coastline retreated between the oldest and most recent shoreline; positive values indicate growth. This statistic does not reflect sub-annual shoreline variability, so will underestimate the full extent of variability at any given location.|
|max_year|Integer|Date|The year that annual shorelines were at their maximum (i.e. located furthest towards the ocean), excluding outliers. This statistic excludes sub-annual shoreline variability.|
|min_year|Integer|Date|The year that annual shorelines were at their minimum (i.e. located furthest inland), excluding outliers. This statistic excludes sub-annual shoreline variability.|
|certainty|String|-|A column providing important data quality flags for each point in the dataset. For more information, see the Quality tab.|
|id_primary|String|-|The name of the point’s Primary sediment compartment from the Australian Coastal Sediment Compartments framework.|

## Load shapefiles

Shapefile polygons should have include 'name' and 'geometry' columns

In [4]:
# Set study area from vector file

# The workflow in this cell combines polygons from two input shapefiles

regions_gdf1 = (
    gpd.read_file(
        '~/gdata1/projects/coastal/SoE/Torres_Strait_2025/Torres Strait Protected Zone.shp' ## Greater Torres Strait area
        
    )
)

regions_gdf2 = (
    gpd.read_file(
        '~/gdata1/projects/coastal/SoE/Torres_Strait_2025/TS Island Coastline Analysis boundaries.shp' ## Greater Torres Strait area
        
    )
)

# combine polygons into single geodataframe
regions_gdf = pd.concat([regions_gdf2, gpd.GeoDataFrame(regions_gdf1.to_crs(regions_gdf2.crs))])
regions_gdf.loc[regions_gdf['name'].isna(), 'name'] = 'Torres Strait'

study_area = regions_gdf

study_area

,Id,name,geometry
0,0,Boigu,"POLYGON ((142.21655 -9.22601, 142.22660 -9.226..."
1,0,Saibai,"POLYGON ((142.79838 -9.36651, 142.80358 -9.370..."
2,0,Dauan,"POLYGON ((142.55273 -9.42149, 142.55029 -9.423..."
3,0,Waral Kawa,"POLYGON ((141.57438 -9.51246, 141.57527 -9.514..."
4,0,Awail Kawa,"POLYGON ((141.56899 -9.60609, 141.57193 -9.606..."
5,0,Mabuiag,"POLYGON ((142.19717 -9.95251, 142.19421 -9.953..."
6,0,Badu,"POLYGON ((142.17142 -10.06306, 142.17265 -10.0..."
7,0,Mua,"POLYGON ((142.26095 -10.12127, 142.26047 -10.1..."
8,0,Warraber,"POLYGON ((142.82937 -10.20402, 142.83078 -10.2..."
9,0,Poruma,"POLYGON ((143.06883 -10.04832, 143.07186 -10.0..."


## Load rates_of_change data, calculate epochs, tabularise and export polygon summaries

In [5]:
!pwd

/home/jovyan/dev/dea-notebooks


In [8]:
baselines

[2025, 2020, 2015, 2010, 2005, 2000]

In [9]:
import geopandas as gpd
import pandas as pd
import numpy as np

from tqdm import tqdm

import fiona

def regression_improvement (gdf, epoch):
    # Get all dist_ columns
    dist_cols = [col for col in gdf.columns if col.startswith('dist_')]
    dist_cols.sort(key=lambda x: int(x.split('_')[1]))

    # Process all rows with progress bar
    print("\nProcessing features...")
    tqdm.pandas()
    results = gdf.progress_apply(lambda row: process_row(row, epoch, dist_cols), axis=1)
    
    # Add results to geopandas dataframe
    gdf['r2_linear'] = results['r2_linear']
    gdf['r2_cubic'] = results['r2_cubic']
    # gdf['is_nonlinear_0.5'] = results['is_nonlinear']
    
    # Calculate R² difference
    gdf['r2_improvement'] = gdf['r2_cubic'] - gdf['r2_linear']
    gdf['extreme_nonlinear'] = gdf['r2_improvement'] > 0.5

    return gdf

In [14]:

for bl_year in baselines:
    print (f'Processing epochs: {bl_year-4}-{bl_year} and 1988-{bl_year}')
    ## Load DEA Coastlines rates of change data
    path = f'../dea-coastlines/data/processed/development_TSRA_{bl_year}_bl/coastlines_development_TSRA_{bl_year}_bl.gpkg'
   
    
    ## Load rates of change data into polygons
    study_area_albers = study_area.to_crs("EPSG:3577")
    
    ratesofchange1 = {}
    
    for x in range(0, len(study_area)):
    
        key = study_area.iloc[x]['name']
        print(f'Sampling rate change points into polygon {x +1} of {len(study_area)}, {key}')
        
        # Load data from coastlines reanalysis for study area bounding box
        bbox = study_area.geometry.bounds.values[x]
        # bbox = study_area.geometry.bounds#.values[0]
        ratesofchange_gdf = get_coastlines1( 
            response=path,
            layer="rates_of_change",
            bbox=tuple(bbox) 
        )


        
        # Add to dictionary
        value = ratesofchange_gdf
        ratesofchange1[key] = value
    
        # Rename columns for consistency
        layers = ['rate_time', 'sig_time', 'se_time', 'outl_time']
        renamed = {}
        for name in layers:
            renamed[name] = f'{name}_1988-{bl_year}'
        
        
        ratesofchange1[key].rename(columns=renamed, inplace=True)
       
    
    # Drop polygons with no rate change points
    ratesofchange = {island: df for island, df in ratesofchange1.items() 
                     # if 'uid' in df.columns}
                     if len(df)>0}

    
    dropped = []
    for key in ratesofchange1.keys():
        if key not in ratesofchange.keys():
            dropped.append(key)
    print ('-----')
    print (f'Excluded polgyons due to absence of change rate data: {dropped}')

    ##Calculate short term change rates for a given baseline year for multiple polygons
    baseline = bl_year
    start_years=[bl_year-4, 1988]
    
    change_epochs = {}
    
    for year in start_years:
        key1 = f'{year}-{baseline}' 
        change_epochs[key1]={}
        for island in list(ratesofchange.keys()):
            if len(ratesofchange[island])>0:
                print (f'{year}, {island}')
                if year==1988:
                    # Use long-term rates of change calculated in DEA Coastlines CLI processing (Step 2)
                    change_epochs[key1][f'{island}'] = ratesofchange[island]
                    # Calculate non-linearity
                    change_epochs[key1][f'{island}'] = regression_improvement(change_epochs[key1][f'{island}'], epoch=f'{year}-{bl_year}')

                else:
                    # Calculate short-term rates of change and append certainty
                    change_epochs[key1][f'{island}'] = calculate_regressions(ratesofchange[island],
                                         year, baseline)
                    change_epochs[key1][f'{island}'] = regression_improvement(change_epochs[key1][f'{island}'], epoch=f'{year}-{bl_year}')
                    change_epochs[key1][f'{island}']['certainty']=ratesofchange[island]['certainty']

           
            if island == 'Torres Strait':
                #Add  rate change points to the master geopackage
                change_epochs[key1][f'{island}'].to_file(path, layer=f'rates_of_change_{key1}', driver='GPKG')


    ## Prepare data for summarising. Select "good" certainty points and apply bias-correction
    for key in change_epochs.keys():
        for key1 in change_epochs[key].keys():
              
            # Optional: Keep only rates of change points with "good" certainty 
            # (i.e. no poor quality flags)
            change_epochs[key][key1] = change_epochs[key][key1].query("certainty == 'good'").copy()
              
            # Get the column names
            rate_cols = change_epochs[key][key1].columns[change_epochs[key][key1].columns.str.startswith('rate_time')]
            sig_cols = change_epochs[key][key1].columns[change_epochs[key][key1].columns.str.startswith('sig_time')]    
            
            # Optional: Apply correction factor from Bishop-Taylor et al. 2021
            change_epochs[key][key1].loc[:, rate_cols] = change_epochs[key][key1].loc[:, rate_cols] + 0.08
                
    # Drop columns with no "good" certainty rates of change
    pop = []
    
    for key in change_epochs.keys():
        for key1 in change_epochs[key].keys():
            if len(change_epochs[key][key1])==0:
                pop.append((key,key1))
                print(f'{key1} {key} removed due to low certainty rate change points')
    for k in pop:
        change_epochs[k[0]].pop(k[-1],None)

    # Long and short statistical summaries for multiple polgyons
    
    summary_df = pd.DataFrame()
    for key in change_epochs.keys():
        for key1 in change_epochs[key].keys():
            # Load point data
            coastlines_data = change_epochs[key][key1][[f'rate_time_{key}', f'sig_time_{key}', 'r2_improvement','geometry']]
            coastlines_data = coastlines_data.rename(columns={f'rate_time_{key}' : 'rate_time'})
            coastlines_data = coastlines_data.rename(columns={f'sig_time_{key}' : 'sig_time'})
            
            # Compute summaries of change for all regions
            summary_df[f'{key1} {key}'] = change_summary(df=coastlines_data, sig=0.01)
    
    # # Sort into pretty format
    summary_df = summary_df.loc[[
        'dynamic', 'nonlinear', 'stable', 'eroding_0.0', 'eroding_0.5', 'eroding_1.0',
        'eroding_3.0', 'eroding_5.0', 'prograd_0.0', 'prograd_0.5', 'prograd_1.0',
        'prograd_3.0', 'prograd_5.0'
    ]]
    
    # Rename index
    summary_df.index = ['Dynamic', 'Non-linear', 'Stable', 
                        'Eroding',     
                        '    > 0.5 m / year', '    > 1.0 m / year', 
                        '    > 3.0 m / year', '    > 5.0 m / year', 
                        'Prograding', 
                        '    > 0.5 m / year', '    > 1.0 m / year', 
                        '    > 3.0 m / year', '    > 5.0 m / year']
    
    # Scale and round
    summary_df = np.round((summary_df * 100),2)
    
    # reorder column names
    gross = [f'Torres Strait {start_years[-1]}-{baseline}',
           f'Torres Strait {start_years[0]}-{baseline}']
    sort = list(summary_df.columns.sort_values())
    for val in gross:
        if val in sort:
            sort.remove(val)
    
    summary_df=summary_df[gross+sort]
    summary_df
    
    print (summary_df)

    ## Export summary_df

    summary_df.to_csv(f"/home/jovyan/dev/dea-notebooks/Testing/TS_Islands_{baseline}_baseline_long_and_short_epochs_v2.csv")
    print (f"Exporting summary table to /home/jovyan/dev/dea-notebooks/Testing/TS_Islands_{baseline}_baseline_long_and_short_epochs_v2.csv")

Processing epochs: 2021-2025 and 1988-2025
Sampling rate change points into polygon 1 of 17, Boigu
Sampling rate change points into polygon 2 of 17, Saibai
Sampling rate change points into polygon 3 of 17, Dauan
Sampling rate change points into polygon 4 of 17, Waral Kawa
Sampling rate change points into polygon 5 of 17, Awail Kawa
Sampling rate change points into polygon 6 of 17, Mabuiag
Sampling rate change points into polygon 7 of 17, Badu
Sampling rate change points into polygon 8 of 17, Mua
Sampling rate change points into polygon 9 of 17, Warraber
Sampling rate change points into polygon 10 of 17, Poruma
Sampling rate change points into polygon 11 of 17, Iama
Sampling rate change points into polygon 12 of 17, Masig
Sampling rate change points into polygon 13 of 17, Ugar
Sampling rate change points into polygon 14 of 17, Erub
Sampling rate change points into polygon 15 of 17, Mer
Sampling rate change points into polygon 16 of 17, Maizab Kaur
Sampling rate change points into polygo

100%|██████████| 1402/1402 [00:00<00:00, 1727.34it/s]


2021, Saibai
Index(['dist_2021', 'dist_2022', 'dist_2023', 'dist_2024', 'dist_2025'], dtype='object')

Processing features...


100%|██████████| 2297/2297 [00:01<00:00, 1876.24it/s]


2021, Dauan
Index(['dist_2021', 'dist_2022', 'dist_2023', 'dist_2024', 'dist_2025'], dtype='object')

Processing features...


...

1988, Mer

Processing features...


100%|██████████| 149/149 [00:00<00:00, 1997.84it/s]


1988, Torres Strait

Processing features...


100%|██████████| 17119/17119 [00:09<00:00, 1886.62it/s]


                    Torres Strait 1988-2000  Torres Strait 1996-2000  \
Dynamic                               12.97                     3.10   
Non-linear                            17.09                    44.06   
Stable                                69.94                    52.84   
Eroding                                2.03                     1.61   
    > 0.5 m / year                     1.64                     1.58   
    > 1.0 m / year                     0.69                     1.40   
    > 3.0 m / year                     0.05                     0.42   
    > 5.0 m / year                     0.01                     0.14   
Prograding                            10.94                     1.49   
    > 0.5 m / year                    10.51                     1.43   
    > 1.0 m / year                     7.54                     1.29   
    > 3.0 m / year                     1.62                     0.29   
    > 5.0 m / year                     0.45                     

In [15]:
# # TESTING


# for bl_year in baselines:
#     # print (f'Processing epochs: {bl_year-4}-{bl_year} and 1988-{bl_year}')
#     # ## Load DEA Coastlines rates of change data
#     # path = f'../dea-coastlines/data/processed/development_TSRA_{bl_year}_bl/coastlines_development_TSRA_{bl_year}_bl.gpkg'
   
    
#     # ## Load rates of change data into polygons
#     # study_area_albers = study_area.to_crs("EPSG:3577")
    
#     # ratesofchange1 = {}
    
#     # for x in range(0, len(study_area)):
    
#     #     key = study_area.iloc[x]['name']
#     #     print(f'Sampling rate change points into polygon {x +1} of {len(study_area)}, {key}')
        
#     #     # Load data from coastlines reanalysis for study area bounding box
#     #     bbox = study_area.geometry.bounds.values[x]
#     #     # bbox = study_area.geometry.bounds#.values[0]
#     #     ratesofchange_gdf = get_coastlines1( 
#     #         response=path,
#     #         layer="rates_of_change",
#     #         bbox=tuple(bbox) 
#     #     )


        
#     #     # Add to dictionary
#     #     value = ratesofchange_gdf
#     #     ratesofchange1[key] = value
    
#     #     # Rename columns for consistency
#     #     layers = ['rate_time', 'sig_time', 'se_time', 'outl_time']
#     #     renamed = {}
#     #     for name in layers:
#     #         renamed[name] = f'{name}_1988-{bl_year}'
        
        
#     #     ratesofchange1[key].rename(columns=renamed, inplace=True)
       
    
#     # # Drop polygons with no rate change points
#     # ratesofchange = {island: df for island, df in ratesofchange1.items() 
#     #                  # if 'uid' in df.columns}
#     #                  if len(df)>0}

    
#     # dropped = []
#     # for key in ratesofchange1.keys():
#     #     if key not in ratesofchange.keys():
#     #         dropped.append(key)
#     # print ('-----')
#     # print (f'Excluded polgyons due to absence of change rate data: {dropped}')

#     # ##Calculate short term change rates for a given baseline year for multiple polygons
#     # baseline = bl_year
#     # start_years=[bl_year-4, 1988]
    
#     # change_epochs = {}
    
#     # for year in start_years:
#     #     key1 = f'{year}-{baseline}' 
#     #     change_epochs[key1]={}
#     #     for island in list(ratesofchange.keys()):
#     #         if len(ratesofchange[island])>0:
#     #             print (f'{year}, {island}')
#     #             if year==1988:
#     #                 # Use long-term rates of change calculated in DEA Coastlines CLI processing (Step 2)
#     #                 change_epochs[key1][f'{island}'] = ratesofchange[island]
#     #                 # Calculate non-linearity
#     #                 change_epochs[key1][f'{island}'] = regression_improvement(change_epochs[key1][f'{island}'], epoch=f'{year}-{bl_year}')

#     #             else:
#     #                 # Calculate short-term rates of change and append certainty
#     #                 change_epochs[key1][f'{island}'] = calculate_regressions(ratesofchange[island],
#     #                                      year, baseline)
#     #                 change_epochs[key1][f'{island}'] = regression_improvement(change_epochs[key1][f'{island}'], epoch=f'{year}-{bl_year}')
#     #                 change_epochs[key1][f'{island}']['certainty']=ratesofchange[island]['certainty']

           
#     #         if island == 'Torres Strait':
#     #             #Add  rate change points to the master geopackage
#     #             change_epochs[key1][f'{island}'].to_file(path, layer=f'rates_of_change_{key1}', driver='GPKG')

#     #     ##Temp
#     # key = ['2016-2020']
#     # key1 = ['Saibai']

#     # for key in key:
#     #     for key1 in key1:
#     # ### End Temp

    
#     ## Prepare data for summarising. Select "good" certainty points and apply bias-correction
#     for key in change_epochs.keys():                   
#         for key1 in change_epochs[key].keys():         
              
#             # Optional: Keep only rates of change points with "good" certainty 
#             # (i.e. no poor quality flags)
#             change_epochs[key][key1] = change_epochs[key][key1].query("certainty == 'good'").copy()
              
#             # Get the column names
#             rate_cols = change_epochs[key][key1].columns[change_epochs[key][key1].columns.str.startswith('rate_time')]
#             sig_cols = change_epochs[key][key1].columns[change_epochs[key][key1].columns.str.startswith('sig_time')]    
            
#             # Optional: Apply correction factor from Bishop-Taylor et al. 2021
#             change_epochs[key][key1].loc[:, rate_cols] = change_epochs[key][key1].loc[:, rate_cols] + 0.08
                
#     # Drop columns with no "good" certainty rates of change
#     pop = []  
    
#     for key in change_epochs.keys():
#         for key1 in change_epochs[key].keys():
#             if len(change_epochs[key][key1])==0:
#                 pop.append((key,key1))
#                 print(f'{key1} {key} removed due to low certainty rate change points')
#     for k in pop:
#         change_epochs[k[0]].pop(k[-1],None)

#     # Long and short statistical summaries for multiple polgyons
    
#     summary_df = pd.DataFrame()
#     for key in change_epochs.keys():
#         for key1 in change_epochs[key].keys():
#             # Load point data
#             coastlines_data = change_epochs[key][key1][[f'rate_time_{key}', f'sig_time_{key}', 'r2_improvement','geometry']]
#             coastlines_data = coastlines_data.rename(columns={f'rate_time_{key}' : 'rate_time'})
#             coastlines_data = coastlines_data.rename(columns={f'sig_time_{key}' : 'sig_time'})
            
#             # Compute summaries of change for all regions
#             summary_df[f'{key1} {key}'] = change_summary(df=coastlines_data, sig=0.01)
    
#     # # Sort into pretty format
#     summary_df = summary_df.loc[[
#         'dynamic', 'nonlinear', 'stable', 'eroding_0.0', 'eroding_0.5', 'eroding_1.0',
#         'eroding_3.0', 'eroding_5.0', 'prograd_0.0', 'prograd_0.5', 'prograd_1.0',
#         'prograd_3.0', 'prograd_5.0'
#     ]]
    
#     # Rename index
#     summary_df.index = ['Dynamic', 'Non-linear', 'Stable', 
#                         'Eroding',     
#                         '    > 0.5 m / year', '    > 1.0 m / year', 
#                         '    > 3.0 m / year', '    > 5.0 m / year', 
#                         'Prograding', 
#                         '    > 0.5 m / year', '    > 1.0 m / year', 
#                         '    > 3.0 m / year', '    > 5.0 m / year']
    
#     # Scale and round
#     summary_df = np.round((summary_df * 100),2)
    
#     # reorder column names
#     gross = [f'Torres Strait {start_years[-1]}-{baseline}',
#            f'Torres Strait {start_years[0]}-{baseline}']
#     sort = list(summary_df.columns.sort_values())
#     for val in gross:
#         if val in sort:
#             sort.remove(val)
    
#     summary_df=summary_df[gross+sort]
#     summary_df
    
#     print (summary_df)

#     ## Export summary_df

#     # summary_df.to_csv(f"/home/jovyan/dev/dea-notebooks/Testing/TS_Islands_{baseline}_baseline_long_and_short_epochs_v2.csv")
#     # print (f"Exporting summary table to /home/jovyan/dev/dea-notebooks/Testing/TS_Islands_{baseline}_baseline_long_and_short_epochs_v2.csv")

In [16]:
# change_epochs['2016-2020']['Saibai']

# Step 5: Prepare single master geopackage

In [17]:
data = {}
output_filename = 'TSRA_v2-1'

for bl_year in baselines:
    path = f'../dea-coastlines/data/processed/development_TSRA_{bl_year}_bl/coastlines_development_TSRA_{bl_year}_bl.gpkg'
    layers= ['shorelines_annual', f'rates_of_change_{bl_year-4}-{bl_year}', f'rates_of_change_1988-{bl_year}']
    data[path]=layers
        
for path, layers in data.items():  
    for layer in layers:
        if layer=='shorelines_annual':
            gdf9 = gpd.read_file(path, layer=layer)
            gdf9.to_file(f'Testing/{output_filename}.gpkg', layer=f'shorelines_annual_1988-{path.split(".gpkg")[0].split("_")[-2]}', driver='GPKG')
        else:
            gdf9 = gpd.read_file(path, layer=layer)
            gdf9.to_file(f'Testing/{output_filename}.gpkg', layer=layer, driver='GPKG')

# Confirm layers in output geopackage
fiona.listlayers(f'Testing/{output_filename}.gpkg')

['shorelines_annual_1988-2025',
 'rates_of_change_2021-2025',
 'rates_of_change_1988-2025',
 'shorelines_annual_1988-2020',
 'rates_of_change_2016-2020',
 'rates_of_change_1988-2020',
 'shorelines_annual_1988-2015',
 'rates_of_change_2011-2015',
 'rates_of_change_1988-2015',
 'shorelines_annual_1988-2010',
 'rates_of_change_2006-2010',
 'rates_of_change_1988-2010',
 'shorelines_annual_1988-2005',
 'rates_of_change_2001-2005',
 'rates_of_change_1988-2005',
 'shorelines_annual_1988-2000',
 'rates_of_change_1996-2000',
 'rates_of_change_1988-2000']

In [32]:
# ratesofchange#_2016-2020